In [1]:
from mdcrow import MDCrow
from langchain.callbacks import get_openai_callback
from datetime import datetime
import os 
import traceback

In [2]:
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.append(parent_dir)
from robustness_prompts import get_prompt # noqa: E402

prompt_6_natural = get_prompt("natural", 6)

prompt_6_natural

'Simulate 1LYZ for 1ps at 300 K. Report the secondary structure assignments of the downloaded PDB structure, and compute the RMSD, SASA, and radius of gyration of the simulation.'

In [3]:
llm_model = "gpt-4-turbo-2024-04-09"
tools = "all"

The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.


In [4]:
agent = MDCrow(
    agent_type="Structured", 
    model=llm_model, 
    top_k_tools=tools, 
    use_memory=False,
    streaming=False,
    verbose=True,
)
with get_openai_callback() as cb:
    chat_start = datetime.now()
    try:
        response = agent.run(prompt_6_natural, callbacks=[cb])
        print(response)
    except Exception as e:
        exc_type, exc_value, exc_traceback = sys.exc_info()
        print(f'{type(e).__name__}:{e}')
        print("".join(traceback.format_exception(exc_type, exc_value, exc_traceback)))
    chat_end = datetime.now()
    total_runtime = (chat_end - chat_start).total_seconds()
    print(cb)
    print(f"Total runtime: {total_runtime:.2f}s")



> Entering new AgentExecutor chain...


Thought: To address the question, I need to first download the PDB structure for the protein 1LYZ, then set up and run a molecular dynamics simulation for 1 picosecond at 300 K. After the simulation, I will compute the secondary structure assignments, RMSD, SASA, and radius of gyration. I will start by downloading the PDB file for 1LYZ.

Action:
```json
{
    "action": "PDBFileDownloader",
    "action_input": {
        "query": "1LYZ"
    }
}
```PDB file found with this ID: 1LYZ

Observation: Succeeded. Downloaded the PDB file:1LYZ_214144
Thought:

Now that I have the PDB file for 1LYZ, the next step is to set up and run a molecular dynamics simulation for 1 picosecond at 300 K. I will use the SetUpandRunFunction tool to perform this task, specifying the necessary parameters for the simulation.

Action:
```json
{
    "action": "SetUpandRunFunction",
    "action_input": {
        "pdb_id": "1LYZ_214144",
        "forcefield_files": ["amber99sb"],
        "system_params": {
            "nonbondedMethod": "CutoffPeriodic",
            "nonbondedCutoff": "1 * nanometers",
            "constraints": "HBonds",
            "rigidWater": true,
            "solvate": true
        },
        "integrator_params": {
            "integrator_type": "LangevinMiddle",
            "Temperature": "300 * kelvin",
            "Friction": "1.0 / picoseconds",
            "Timestep": "0.002 * picoseconds"
        },
        "simulation_params": {
            "Ensemble": "NVT",
            "Number of Steps": 500,
            "record_interval_steps": 5


To address the simulation setup failure, I need to correct the forcefield file name and specify the constraintTolerance parameter since rigidWater is set to True. I will use the 'amber99sb.xml' forcefield file from the available options and set a constraintTolerance. I will then rerun the simulation setup.

Action:
```json
{
    "action": "SetUpandRunFunction",
    "action_input": {
        "pdb_id": "1LYZ_214144",
        "forcefield_files": ["amber99sb.xml"],
        "system_params": {
            "nonbondedMethod": "CutoffPeriodic",
            "nonbondedCutoff": "1 * nanometers",
            "constraints": "HBonds",
            "rigidWater": true,
            "solvate": true,
            "constraintTolerance": "0.00001"
        },
        "integrator_params": {
            "integrator_type": "LangevinMiddle",
            "Temperature": "300 * kelvin",
            "Friction": "1.0 / picoseconds",
            "Timestep": "0.002 * picoseconds"
        },
        "simulation_params": {

About to create system...
Attempts at creating system: 0/3
Error adding solvent ValueError – No template found for residue 1 (LYS).  The set of atoms is similar to HID, but it is missing 8 atoms.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#template
Residue code: LYS
Residue code not in solvent list. Adding forcefield                         not supported.

Observation: Failed. No template found for residue 1 (LYS).  The set of atoms is similar to HID, but it is missing 8 atoms.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#templateThis were the inputs {'pdb_id': '1LYZ_214144', 'forcefield_files': ['amber99sb.xml'], 'system_params': {'nonbondedMethod': 'CutoffPeriodic', 'nonbondedCutoff': '1 * nanometers', 'constraints': 'HBonds', 'rigidWater': True, 'solvate': True, 'constraintTolerance': '0.00001'}, 'integrator_params': {'integrator_type': 'LangevinMiddle', 'Temperature': '300 * kelvin', 'Fri

...


Observation: Succeeded. File cleaned! 
File ID: 1LYZ_214234
Thought:

Now that the PDB file has been successfully cleaned and prepared without water molecules, I will attempt to set up and run the molecular dynamics simulation again using the corrected parameters and the cleaned PDB file. This should address the previous issues encountered with missing atoms and forcefield compatibility.

Action:
```json
{
    "action": "SetUpandRunFunction",
    "action_input": {
        "pdb_id": "1LYZ_214234",
        "forcefield_files": ["amber99sb.xml"],
        "system_params": {
            "nonbondedMethod": "CutoffPeriodic",
            "nonbondedCutoff": "1 * nanometers",
            "constraints": "HBonds",
            "rigidWater": true,
            "solvate": true,
            "constraintTolerance": "0.00001"
        },
        "integrator_params": {
            "integrator_type": "LangevinMiddle",
            "Temperature": "300 * kelvin",
            "Friction": "1.0 / picoseconds",
            "Timestep": "0.002 * picoseconds"
        },
        "simulati

About to create system...
Attempts at creating system: 0/3


Error adding solvent ValueError – No template found for residue 130 (HOH).  This might mean your input topology is missing some atoms or bonds, or possibly that you are using the wrong force field.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#template
Residue code: HOH
Trying to add missing component to Forcefield...
No SMILES found for HET code.

Observation: Failed. No template found for residue 130 (HOH).  This might mean your input topology is missing some atoms or bonds, or possibly that you are using the wrong force field.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#templateThis were the inputs {'pdb_id': '1LYZ_214234', 'forcefield_files': ['amber99sb.xml'], 'system_params': {'nonbondedMethod': 'CutoffPeriodic', 'nonbondedCutoff': '1 * nanometers', 'constraints': 'HBonds', 'rigidWater': True, 'solvate': True, 'constraintTolerance': '0.00001'}, 'integrator_params': {'integrator_type': 'L

Final Answer: After multiple attempts to clean the PDB file and set up the molecular dynamics simulation for 1LYZ, persistent issues with missing templates for water molecules (HOH) have prevented the successful execution of the simulation. Despite repeated cleaning of the file to remove water and heterogens, the simulation setup tool continues to report errors related to the presence of water molecules. This suggests a deeper compatibility issue between the cleaned PDB file and the forcefield or a possible error in recognizing the changes made during the cleaning process. Further investigation and possibly manual adjustments to the PDB file or the use of a different simulation setup tool or parameters might be necessary to resolve these issues and successfully simulate 1LYZ.

> Finished chain.
({'input': '\n    You are an expert molecular dynamics scientist, and\n    your task is to respond to the question or\n    solve the problem to the best of your ability using\n    the provided t

In [5]:
registry = agent.path_registry
print(registry.list_path_names_and_descriptions().replace(",", "\n"))

Files found in registry: 1LYZ_214144: PDB file downloaded from RSCB
 PDBFile ID: 1LYZ_214144
 1LYZ_214203: Cleaned File:  Removed Heterogens
 and Water Removed.  Replaced Nonstandard Residues. Added Hydrogens at pH 7.0. Missing Atoms added and nonstandard residues replaced. 
 1LYZ_214219: Cleaned File:  Removed Heterogens
 and Water Removed.  Replaced Nonstandard Residues. Added Hydrogens at pH 7.0. Missing Atoms added and nonstandard residues replaced. 
 1LYZ_214234: Cleaned File:  Removed Heterogens
 and Water Removed.  Replaced Nonstandard Residues. Added Hydrogens at pH 7.0. Missing Atoms added and nonstandard residues replaced. 


In [6]:
# make sure pdb was downloaded
assert os.path.exists(registry.get_mapped_path("1LYZ_214144"))

In [7]:
# # make sure dssp was computed correctly
# from mdcrow.tools.base_tools import ComputeDSSP

# dssp = ComputeDSSP(registry)
# dssp._run(traj_file= "1LYZ_125306", target_frames="first")

In [8]:
# # make sure trajectory and topology exist
# traj_path = registry.get_mapped_path("rec0_125315")
# top_path = registry.get_mapped_path("top_sim0_023025")

# assert os.path.exists(traj_path)
# assert os.path.exists(top_path)

In [9]:
# # make sure rmsd plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_022913'))

In [10]:
# # make sure rgy plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_023033'))

In [11]:
# # make sure sasa plot was generated
# from IPython.display import Image
# Image(filename=registry.get_mapped_path('fig0_023320'))

In [13]:
#verify the total cost
def calculate_llm_cost(input_tokens, output_tokens, model):
    pricing_2024 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 5/1e6, "output": 15/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
        "claude-3-opus": {"input": 15/1e6, "output": 75/1e6},
        "claude-3.5-sonnet": {"input": 3/1e6, "output": 15/1e6},
    }
    pricing_2025 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 2.5/1e6, "output": 10/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
    }
    
    prices = pricing_2025[model]
    cost = (input_tokens * prices["input"]) + (output_tokens * prices["output"])
    return round(cost, 6)

llm_cost = calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name)
print('Input tokens:',cb.prompt_tokens)
print('Output tokens:',cb.completion_tokens)
print('LLM costs: $',llm_cost)

Input tokens: 94046
Output tokens: 2101
LLM costs: $ 1.00349
